# Gaussian Process multi-objective hyperparameter experiment v2: bias-aware

This notebook extends the first GP multi-objective experiment.

Version 1 optimized GP hyperparameters using two objectives:

1. monthly NBS RMSE
2. 6-month integrated NBS RMSE

The results showed that this could slightly improve integrated RMSE, but sometimes increased cumulative bias. Since USACE has identified systematic offset/drift in cumulative NBS as a key concern, this version adds a third objective:

3. mean absolute lake-level 6-month integrated NBS bias

The purpose is to search for GP hyperparameters that preserve monthly and integrated skill while reducing cumulative drift.

## 1. Install optional dependency

In [ ]:
# Uncomment if needed.
# !pip install platypus-opt

## 2. Imports and setup

In [ ]:
import os
import sys
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RationalQuadratic
from sklearn.metrics import mean_squared_error, r2_score

from platypus import NSGAII, Problem, Real

import joblib

In [ ]:
sys.path.append(os.path.abspath("../../"))

from src.data_processor import CFSTransformer
from src.data_loader import DataLoader

## 3. User inputs

In [ ]:
local_path = "/Users/dannes/Documents/cnbs-predictor-dev/"
input_dir = local_path + "cnbs-predictor/data/"

train_start_date = "1981-09-01"
train_end_date = "2005-12-01"

val_start_date = "2006-01-01"
val_end_date = "2010-03-01"

random_state = 42

lakes = ["superior", "michigan-huron", "erie", "ontario"]

# Keep this modest at first. GP fitting is expensive.
nsgaii_evaluations = 30

## 4. Load CFSR features

In [ ]:
pcp_data = pd.read_csv(input_dir + "cfsr/CFSR_APCP_Basin_Avgs.csv")
evap_data = pd.read_csv(input_dir + "cfsr/CFSR_EVAP_Basin_Avgs.csv")
tmp_data = pd.read_csv(input_dir + "cfsr/CFSR_TMP_Basin_Avgs.csv")

pcp_data["date"] = pd.to_datetime(pcp_data[["year", "month"]].assign(day=1))

X = pd.DataFrame({
    "superior_lake_precipitation": pcp_data["sup_lake"],
    "michigan-huron_lake_precipitation": pcp_data["mih_lake"],
    "erie_lake_precipitation": pcp_data["eri_lake"],
    "ontario_lake_precipitation": pcp_data["ont_lake"],

    "superior_land_precipitation": pcp_data["sup_land"],
    "michigan-huron_land_precipitation": pcp_data["mih_land"],
    "erie_land_precipitation": pcp_data["eri_land"],
    "ontario_land_precipitation": pcp_data["ont_land"],

    "superior_lake_evaporation": evap_data["sup_lake"],
    "michigan-huron_lake_evaporation": evap_data["mih_lake"],
    "erie_lake_evaporation": evap_data["eri_lake"],
    "ontario_lake_evaporation": evap_data["ont_lake"],

    "superior_land_evaporation": evap_data["sup_land"],
    "michigan-huron_land_evaporation": evap_data["mih_land"],
    "erie_land_evaporation": evap_data["eri_land"],
    "ontario_land_evaporation": evap_data["ont_land"],

    "superior_lake_air_temperature": tmp_data["sup_lake"],
    "michigan-huron_lake_air_temperature": tmp_data["mih_lake"],
    "erie_lake_air_temperature": tmp_data["eri_lake"],
    "ontario_lake_air_temperature": tmp_data["ont_lake"],

    "superior_land_air_temperature": tmp_data["sup_land"],
    "michigan-huron_land_air_temperature": tmp_data["mih_land"],
    "erie_land_air_temperature": tmp_data["eri_land"],
    "ontario_land_air_temperature": tmp_data["ont_land"],
})

X.set_index(pd.to_datetime(pcp_data[["year", "month"]].assign(day=1)), inplace=True)

X_shifted = CFSTransformer(X).shift_variables(lag=0, lead=10)

X_shifted = pd.concat(
    [
        X_shifted,
        pd.DataFrame({"month": X_shifted.index.month}, index=X_shifted.index),
    ],
    axis=1,
)

X_shifted = pd.get_dummies(X_shifted, columns=["month"], prefix="month")

feature_column_order = (
    [f"month_{i}" for i in range(1, 13)]
    + [
        f"{lake}_{surface_type}_{comp}_mo{m}"
        for lake in lakes
        for surface_type in ["lake", "land"]
        for comp in ["precipitation", "evaporation", "air_temperature"]
        for m in range(10)
    ]
)

X_reorg = X_shifted.reindex(columns=feature_column_order)

dataloader = DataLoader()
df_sst_k = dataloader.glsea(input_dir + "glsea/oisst_sst_1981-2024.csv", units="K")

X_merged = pd.merge(df_sst_k, X_reorg, left_index=True, right_index=True, how="inner")

print(X_merged.shape)
X_merged.head()

## 5. Load targets

This mirrors the production training file: precipitation, evaporation, runoff, and NBS are all included as targets. The multi-objective evaluation focuses on NBS.

In [ ]:
l2 = dataloader.l2swbm(input_dir + "l2swbm/")
glcc = dataloader.glcc(input_dir + "glcc/", units="mm")

targets = pd.DataFrame({
    "superior_target_evaporation": l2["superior_evaporation_obs"],
    "superior_target_precipitation": l2["superior_precipitation_obs"],
    "superior_target_runoff": l2["superior_runoff_obs"],
    "superior_target_nbs": glcc["superior_nbs_obs"],

    "michigan-huron_target_evaporation": l2["michigan-huron_evaporation_obs"],
    "michigan-huron_target_precipitation": l2["michigan-huron_precipitation_obs"],
    "michigan-huron_target_runoff": l2["michigan-huron_runoff_obs"],
    "michigan-huron_target_nbs": glcc["michigan-huron_nbs_obs"],

    "erie_target_evaporation": l2["erie_evaporation_obs"],
    "erie_target_precipitation": l2["erie_precipitation_obs"],
    "erie_target_runoff": l2["erie_runoff_obs"],
    "erie_target_nbs": glcc["erie_nbs_obs"],

    "ontario_target_evaporation": l2["ontario_evaporation_obs"],
    "ontario_target_precipitation": l2["ontario_precipitation_obs"],
    "ontario_target_runoff": l2["ontario_runoff_obs"],
    "ontario_target_nbs": glcc["ontario_nbs_obs"],
}).dropna()

targets_lead = CFSTransformer(targets).shift_variables(lag=0, lead=12)

target_column_order = [
    f"{lake}_target_{comp}_mo{m}"
    for lake in lakes
    for comp in ["precipitation", "evaporation", "runoff", "nbs"]
    for m in range(12)
]

targets_reorg = targets_lead.reindex(columns=target_column_order)

aligned_X, aligned_y = X_merged.align(targets_reorg, join="inner", axis=0)

print(f"Number of features: {aligned_X.shape[1]}")
print(f"Number of targets: {aligned_y.shape[1]}")

## 6. Train/test split and scaling

In [ ]:
X_train = aligned_X[train_start_date:train_end_date]
y_train = aligned_y[train_start_date:train_end_date]

X_test = aligned_X[val_start_date:val_end_date]
y_test = aligned_y[val_start_date:val_end_date]

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)

print(X_train_scaled.shape, y_train_scaled.shape)
print(X_test_scaled.shape, y_test_scaled.shape)

## 7. Evaluation helpers

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def unscale_to_df(y_scaled, scaler, index, columns):
    return pd.DataFrame(
        scaler.inverse_transform(y_scaled),
        index=index,
        columns=columns,
    )


def nbs_columns(columns):
    return [c for c in columns if "_target_nbs_mo" in c]


def monthly_nbs_rmse(y_true_df, y_pred_df):
    cols = nbs_columns(y_true_df.columns)
    yt = y_true_df[cols].to_numpy().ravel()
    yp = y_pred_df[cols].to_numpy().ravel()
    return rmse(yt, yp)


def lake_window_values(y_true_df, y_pred_df, lake, window=6):
    cols = [f"{lake}_target_nbs_mo{i}" for i in range(12)]
    yt = y_true_df[cols].to_numpy()
    yp = y_pred_df[cols].to_numpy()

    values_true = []
    values_pred = []

    for start in range(12 - window + 1):
        values_true.append(yt[:, start:start + window].sum(axis=1))
        values_pred.append(yp[:, start:start + window].sum(axis=1))

    return np.concatenate(values_true), np.concatenate(values_pred)


def six_month_integrated_nbs_rmse(y_true_df, y_pred_df):
    values_true = []
    values_pred = []

    for lake in lakes:
        yt, yp = lake_window_values(y_true_df, y_pred_df, lake, window=6)
        values_true.append(yt)
        values_pred.append(yp)

    yt_all = np.concatenate(values_true)
    yp_all = np.concatenate(values_pred)

    return rmse(yt_all, yp_all)


def six_month_integrated_nbs_bias(y_true_df, y_pred_df):
    values_true = []
    values_pred = []

    for lake in lakes:
        yt, yp = lake_window_values(y_true_df, y_pred_df, lake, window=6)
        values_true.append(yt)
        values_pred.append(yp)

    yt_all = np.concatenate(values_true)
    yp_all = np.concatenate(values_pred)

    return np.mean(yp_all - yt_all)


def lake_level_six_month_biases(y_true_df, y_pred_df):
    out = {}

    for lake in lakes:
        yt, yp = lake_window_values(y_true_df, y_pred_df, lake, window=6)
        out[lake] = np.mean(yp - yt)

    return out


def mean_abs_lake_level_six_month_bias(y_true_df, y_pred_df):
    biases = lake_level_six_month_biases(y_true_df, y_pred_df)
    return np.mean(np.abs(list(biases.values())))


def max_abs_lake_level_six_month_bias(y_true_df, y_pred_df):
    biases = lake_level_six_month_biases(y_true_df, y_pred_df)
    return np.max(np.abs(list(biases.values())))


def evaluate_prediction(y_pred_scaled, label):
    y_true_df = unscale_to_df(y_test_scaled, y_scaler, y_test.index, y_test.columns)
    y_pred_df = unscale_to_df(y_pred_scaled, y_scaler, y_test.index, y_test.columns)

    lake_biases = lake_level_six_month_biases(y_true_df, y_pred_df)

    metrics = {
        "label": label,
        "monthly_nbs_rmse": monthly_nbs_rmse(y_true_df, y_pred_df),
        "six_month_integrated_nbs_rmse": six_month_integrated_nbs_rmse(y_true_df, y_pred_df),
        "six_month_integrated_nbs_bias": six_month_integrated_nbs_bias(y_true_df, y_pred_df),
        "mean_abs_lake_level_six_month_bias": mean_abs_lake_level_six_month_bias(y_true_df, y_pred_df),
        "max_abs_lake_level_six_month_bias": max_abs_lake_level_six_month_bias(y_true_df, y_pred_df),
        "overall_scaled_r2": r2_score(y_test_scaled, y_pred_scaled),
    }

    for lake, bias in lake_biases.items():
        metrics[f"{lake}_six_month_bias"] = bias

    return metrics

## 8. Production-style GP baseline

In [ ]:
baseline_kernel = 1.0 * Matern(nu=1.5) * RationalQuadratic()

baseline_gp = GaussianProcessRegressor(
    kernel=baseline_kernel,
    alpha=0.1,
    n_restarts_optimizer=10,
    random_state=random_state,
)

baseline_gp.fit(X_train_scaled, y_train_scaled)
y_pred_baseline_scaled = baseline_gp.predict(X_test_scaled)

baseline_metrics = evaluate_prediction(y_pred_baseline_scaled, "production_style_gp")

print("Optimized kernel:", baseline_gp.kernel_)
baseline_metrics

## 9. Bias-aware multi-objective GP candidate evaluator

The outer optimizer proposes GP hyperparameters. For each candidate, we minimize:

1. monthly NBS RMSE
2. 6-month integrated NBS RMSE
3. mean absolute lake-level 6-month integrated NBS bias

The third objective prevents bias cancellation across lakes.

In [ ]:
evaluation_log = []

def build_gp_from_hyperparams(hyperparams):
    matern_length_scale, rq_alpha, rq_length_scale, gp_alpha = hyperparams

    kernel = (
        1.0
        * Matern(length_scale=matern_length_scale, nu=1.5)
        * RationalQuadratic(alpha=rq_alpha, length_scale=rq_length_scale)
    )

    return GaussianProcessRegressor(
        kernel=kernel,
        alpha=gp_alpha,
        optimizer=None,
        normalize_y=False,
        random_state=random_state,
    )


def gp_multiobjective_cost(hyperparams):
    try:
        gp = build_gp_from_hyperparams(hyperparams)
        gp.fit(X_train_scaled, y_train_scaled)
        y_pred_scaled = gp.predict(X_test_scaled)

        metrics = evaluate_prediction(
            y_pred_scaled,
            label="candidate_gp",
        )

        obj_monthly = metrics["monthly_nbs_rmse"]
        obj_cumulative = metrics["six_month_integrated_nbs_rmse"]
        obj_bias = metrics["mean_abs_lake_level_six_month_bias"]

        evaluation_log.append({
            "matern_length_scale": hyperparams[0],
            "rq_alpha": hyperparams[1],
            "rq_length_scale": hyperparams[2],
            "gp_alpha": hyperparams[3],
            **metrics,
        })

        print(
            f"monthly={obj_monthly:.3f}, "
            f"six_month={obj_cumulative:.3f}, "
            f"mean_abs_bias={obj_bias:.3f}, "
            f"global_bias={metrics['six_month_integrated_nbs_bias']:.3f}, "
            f"params={hyperparams}"
        )

        return [obj_monthly, obj_cumulative, obj_bias]

    except Exception as exc:
        print(f"Candidate failed: {hyperparams} -> {exc}")
        return [1e9, 1e9, 1e9]

## 10. Run NSGA-II search

In [ ]:
problem = Problem(4, 3)

problem.types[:] = [
    Real(0.1, 100.0),   # Matern length scale
    Real(0.01, 10.0),   # RationalQuadratic alpha
    Real(0.1, 100.0),   # RationalQuadratic length scale
    Real(1e-4, 1.0),    # GP alpha / noise level
]

problem.function = gp_multiobjective_cost

algorithm = NSGAII(problem)
algorithm.run(nsgaii_evaluations)

## 11. Inspect candidate results

In [ ]:
eval_df = pd.DataFrame(evaluation_log)
eval_df = eval_df.sort_values(
    [
        "mean_abs_lake_level_six_month_bias",
        "six_month_integrated_nbs_rmse",
        "monthly_nbs_rmse",
    ]
).reset_index(drop=True)

eval_df.head(20)

In [ ]:
solutions = algorithm.result

pareto_df = pd.DataFrame(
    [
        {
            "monthly_nbs_rmse": s.objectives[0],
            "six_month_integrated_nbs_rmse": s.objectives[1],
            "mean_abs_lake_level_six_month_bias": s.objectives[2],
            "matern_length_scale": s.variables[0],
            "rq_alpha": s.variables[1],
            "rq_length_scale": s.variables[2],
            "gp_alpha": s.variables[3],
        }
        for s in solutions
    ]
)

pareto_df = pareto_df.sort_values(
    [
        "mean_abs_lake_level_six_month_bias",
        "six_month_integrated_nbs_rmse",
        "monthly_nbs_rmse",
    ]
).reset_index(drop=True)

pareto_df

## 12. Plot objective space

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(
    eval_df["monthly_nbs_rmse"],
    eval_df["six_month_integrated_nbs_rmse"],
    c=eval_df["mean_abs_lake_level_six_month_bias"],
    alpha=0.7,
)

plt.scatter(
    baseline_metrics["monthly_nbs_rmse"],
    baseline_metrics["six_month_integrated_nbs_rmse"],
    marker="*",
    s=180,
    label="production-style GP",
)

plt.xlabel("Monthly NBS RMSE")
plt.ylabel("6-month integrated NBS RMSE")
plt.title("GP candidates colored by mean abs lake-level bias")
plt.colorbar(label="Mean abs lake-level 6-month bias")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(
    eval_df["six_month_integrated_nbs_rmse"],
    eval_df["mean_abs_lake_level_six_month_bias"],
    alpha=0.7,
    label="evaluated candidates",
)

plt.scatter(
    baseline_metrics["six_month_integrated_nbs_rmse"],
    baseline_metrics["mean_abs_lake_level_six_month_bias"],
    marker="*",
    s=180,
    label="production-style GP",
)

plt.xlabel("6-month integrated NBS RMSE")
plt.ylabel("Mean abs lake-level 6-month bias")
plt.title("RMSE-bias tradeoff")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 13. Select a bias-aware candidate

Selection rule:

Choose the candidate with the lowest mean absolute lake-level 6-month bias among models whose 6-month RMSE is within 2% of the production-style GP baseline.

This reflects the operational goal: do not sacrifice much RMSE, but reduce systematic drift.

In [ ]:
rmse_tolerance = 1.02

candidate_pool = eval_df[
    eval_df["six_month_integrated_nbs_rmse"]
    <= baseline_metrics["six_month_integrated_nbs_rmse"] * rmse_tolerance
].copy()

if candidate_pool.empty:
    print("No candidates within RMSE tolerance; selecting lowest bias overall.")
    selected = eval_df.sort_values("mean_abs_lake_level_six_month_bias").iloc[0]
else:
    selected = candidate_pool.sort_values(
        ["mean_abs_lake_level_six_month_bias", "six_month_integrated_nbs_rmse"]
    ).iloc[0]

selected

In [ ]:
selected_hyperparams = [
    selected["matern_length_scale"],
    selected["rq_alpha"],
    selected["rq_length_scale"],
    selected["gp_alpha"],
]

selected_gp = build_gp_from_hyperparams(selected_hyperparams)
selected_gp.fit(X_train_scaled, y_train_scaled)
y_pred_selected_scaled = selected_gp.predict(X_test_scaled)

selected_metrics = evaluate_prediction(y_pred_selected_scaled, "bias_aware_multiobjective_gp")

pd.DataFrame([baseline_metrics, selected_metrics])

## 14. Integrated metrics by lake and window

In [ ]:
def integrated_metrics_by_lake(y_true_df, y_pred_df, windows=(1, 3, 6)):
    rows = []

    for lake in lakes:
        for window in windows:
            yt_all, yp_all = lake_window_values(y_true_df, y_pred_df, lake, window=window)

            rows.append({
                "Lake": lake,
                "Window": window,
                "RMSE": rmse(yt_all, yp_all),
                "R2": r2_score(yt_all, yp_all),
                "Bias": np.mean(yp_all - yt_all),
            })

    return pd.DataFrame(rows)


y_true_df = unscale_to_df(y_test_scaled, y_scaler, y_test.index, y_test.columns)
y_pred_baseline_df = unscale_to_df(y_pred_baseline_scaled, y_scaler, y_test.index, y_test.columns)
y_pred_selected_df = unscale_to_df(y_pred_selected_scaled, y_scaler, y_test.index, y_test.columns)

integrated_baseline = integrated_metrics_by_lake(y_true_df, y_pred_baseline_df)
integrated_baseline["Model"] = "production_style_gp"

integrated_selected = integrated_metrics_by_lake(y_true_df, y_pred_selected_df)
integrated_selected["Model"] = "bias_aware_multiobjective_gp"

integrated_df = pd.concat([integrated_baseline, integrated_selected], ignore_index=True)

integrated_df

In [ ]:
integrated_summary = (
    integrated_df
    .groupby(["Model", "Window"], as_index=False)
    .agg(
        RMSE=("RMSE", "mean"),
        R2=("R2", "mean"),
        Bias=("Bias", "mean"),
        MeanAbsBias=("Bias", lambda x: np.mean(np.abs(x))),
        MaxAbsBias=("Bias", lambda x: np.max(np.abs(x))),
    )
    .sort_values(["Window", "RMSE"])
)

integrated_summary

## 15. Plot integrated RMSE and bias

In [ ]:
plot_df = integrated_summary.pivot(index="Model", columns="Window", values="RMSE")

ax = plot_df.plot(kind="bar", figsize=(9, 5))
ax.set_ylabel("Integrated RMSE")
ax.set_title("Bias-aware GP integrated NBS RMSE by window")
ax.set_xlabel("")
ax.legend(title="Window")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = integrated_summary.pivot(index="Model", columns="Window", values="MeanAbsBias")

ax = plot_df.plot(kind="bar", figsize=(9, 5))
ax.set_ylabel("Mean absolute lake-level bias")
ax.set_title("Bias-aware GP lake-level bias by window")
ax.set_xlabel("")
ax.legend(title="Window")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 16. Notes and interpretation

Questions to answer:

- Does adding lake-level bias as an objective reduce cumulative drift?
- Is the selected candidate within an acceptable RMSE tolerance?
- Does bias improve for all lakes or only some lakes?
- Does the selected candidate reduce mean absolute lake-level bias without increasing max absolute bias?
- Should the next version use `max_abs_lake_level_six_month_bias` instead of mean absolute bias?

In [ ]:
# Optional: save outputs
# output_dir = Path(input_dir) / "analysis_outputs"
# output_dir.mkdir(parents=True, exist_ok=True)
# eval_df.to_csv(output_dir / "gp_bias_aware_candidate_log.csv", index=False)
# pareto_df.to_csv(output_dir / "gp_bias_aware_pareto.csv", index=False)
# integrated_df.to_csv(output_dir / "gp_bias_aware_integrated_by_lake.csv", index=False)
# integrated_summary.to_csv(output_dir / "gp_bias_aware_integrated_summary.csv", index=False)